# Data Processing for Aalto Gear Failure Dataset (AGFD)

- https://www.sciencedirect.com/science/article/pii/S2352340924011338
- https://data.mendeley.com/datasets/fywnj597d8/2

**NOTES:**

`acc2` in installation 3 are pure noice, as the sensor is damaged. DO NOT USE!

**V1**

- Step size 3

**V2**

- Step size 2
- Dropped `sample_index`


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from signal_analysis import FFT_TSA, order_track_FFT, order_track_signal, signal_TSA
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

In [ ]:
# HELPERS


def fix_time(X):
    increment = 1 / 3012  # For 3012 Hz sampling rate
    loc = np.where(X[1:] < X[:-1])[0] + 1  # + 1 needed to get the right index

    if len(loc) == 0:
        return X

    for l in loc:
        wrong = X[l - 1] - X[l]
        print(f"Wrong time at {l}")
        diff = np.concatenate([
            np.zeros(l),
            np.ones(len(X) - l) * (increment + wrong),
        ])
        X = X + diff

    return X

In [ ]:
# SAMPLE OF DATA

data_folder = Path.cwd().parent.parent.parent / "data" / "mendeley_AGFD3kHz"
files = data_folder.glob("**/*.csv")

df = pd.read_csv(next(files), sep=",", index_col=0, header=0)
df.head(3)

In [ ]:
# PROCESSING

torque_map = {
    "0_12Nm": 1,
    "0_71Nm": 6,
    "1_31Nm": 11,
}

# S - Small, L - Large
fault_map = {
    "Mild_Wear_GP7": ("S", "wear"),
    "Severe_Wear_GP2": ("L", "wear"),
    "Mild_TFF_GP9": ("S", "crack"),
    "Severe_TFF_GP5": ("L", "crack"),
    "Mild_Pitting_GP1": ("S", "pitting"),
    "Severe_Pitting_GP6": ("L", "pitting"),
}

# TSA
revolutions_per_TSA = 40
step_size = 2

# Specify raw data
data_folder = Path.cwd().parent.parent.parent / "data" / "mendeley_AGFD3kHz"
files = data_folder.glob("**/*.csv")

# Go through files
dfs = []
for f in files:
    # SPECS
    ##

    # Get measurement specifications from file path
    p = f.parts
    # print(p)

    rpm = int(p[6].replace("RPM", ""))
    torque = torque_map[p[7]]

    if p[8] == "Healthy":
        installation = 0
        level = "-"
        GP = int(p[9][2])
        fault = "healthy"
    elif p[8] == "Faulty":
        installation = int(p[9][-1])
        level, fault = fault_map.get(
            p[10].replace(".csv", ""), (None, None)
        )  # Fault types we don't care about not in map
        GP = 0
    else:
        raise ValueError(f"Unknown type directory: {p[8]}")

    # Skip the faults we don't want
    if fault is None:
        continue

    # SIGNAL
    ##

    # Read CSV
    df = pd.read_csv(f, sep=",", index_col=0, header=0)

    # Installation 3 special case
    if installation == 3:
        # Reset index
        df = df.reset_index(
            drop=False  # ! Installation 3 has time as index. This is done to add separate index, but not drop time
        )

        # Cut first and last 20%, which has already been done for other installations
        df = df.iloc[int(len(df) * 0.2) : int(len(df) * 0.8)].reset_index(drop=True)

    print(str(f))

    # TODO: Save real average torque at faulty gear (Need to find out multiplier, because neither Torq1 or Torq2 are on the right shaft)
    # real_torque_avg = df["Torq1"].mean()

    # These sensors not used in this study
    df.drop(
        columns=[
            "enc1_ang",
            "enc2_ang",
            "enc3_ang",
            "enc5_ang",
            "acc1",
            "Torq1",
            "Torq2",
        ],
        inplace=True,
    )

    # Enc4 runs the wrong way in most files
    if df["enc4_ang"].iloc[0] > df["enc4_ang"].iloc[1]:
        df["enc4_ang"] = -df["enc4_ang"]

    # Some measurements have glitches in time
    df["time"] = fix_time(df["time"].to_numpy())

    # Sometimes last value is NaN, which causes problems later
    if any([np.isnan(df[s].iloc[-1]) for s in ["acc2", "acc3", "acc4"]]):
        df = df.iloc[:-1].reset_index(drop=True)

    # Do OT and TSA
    all_sensor_samples_SIGNAL_OT = []
    all_sensor_samples_FFT_OT = []
    for sensor in ["acc2", "acc3", "acc4"]:
        # SIGNAL OT METHOD
        ###

        # OT
        num_samples_per_revolution = int(3012 / (rpm / 60 / 3)) + 50
        signal_OT = order_track_signal(
            df[sensor].to_numpy(),
            df["time"].to_numpy(),
            df["enc4_ang"].to_numpy(),
            num_samples_per_revolution=num_samples_per_revolution,
        )

        signal_OT_TSA = signal_TSA(
            signal_OT, num_samples_per_revolution, revolutions_per_TSA, step_size
        )
        samples = np.abs(np.fft.rfft(signal_OT_TSA, norm="forward", axis=1))[
            :, : 11 * 15
        ]  # 10 first full GMFs and the DC component

        samples = samples.astype(np.float32)
        all_sensor_samples_SIGNAL_OT.append(samples)

        # FFT OT METHOD
        ###

        revolution_FFTs = order_track_FFT(
            df[sensor].to_numpy(),
            df["time"].to_numpy(),
            df["enc4_ang"].to_numpy(),
            num_orders=11 * 15,
            fs=3012,
        )
        samples = FFT_TSA(revolution_FFTs, revolutions_per_TSA, step_size)
        samples = np.array(samples, dtype=np.float32)

        all_sensor_samples_FFT_OT.append(samples)

    all_sensor_samples_SIGNAL_OT = np.array(all_sensor_samples_SIGNAL_OT)
    all_sensor_samples_FFT_OT = np.array(all_sensor_samples_FFT_OT)

    # Make a new row for the DF for every TSA sample
    expanded_rows = []
    for j in range(all_sensor_samples_SIGNAL_OT.shape[1]):
        expanded_rows.append({
            "speed": rpm,
            "load": torque,
            "installation": installation,
            "severity": level,
            "healthy_GP": GP,
            "class": fault,
            "OT_method": "signal",
            "TSA_size": revolutions_per_TSA,
            "acc2": all_sensor_samples_SIGNAL_OT[0][j],
            "acc3": all_sensor_samples_SIGNAL_OT[1][j],
            "acc4": all_sensor_samples_SIGNAL_OT[2][j],
        })
    for j in range(all_sensor_samples_FFT_OT.shape[1]):
        expanded_rows.append({
            "speed": rpm,
            "load": torque,
            "installation": installation,
            "severity": level,
            "healthy_GP": GP,
            "class": fault,
            "OT_method": "fft",
            "TSA_size": revolutions_per_TSA,
            "acc2": all_sensor_samples_FFT_OT[0][j],
            "acc3": all_sensor_samples_FFT_OT[1][j],
            "acc4": all_sensor_samples_FFT_OT[2][j],
        })

    # Make DF of one file
    dfs.append(pd.DataFrame(expanded_rows))

# Combine files
dfs = pd.concat(dfs)
dfs.head(3)

In [ ]:
dfs.dtypes

In [ ]:
# SAVING

dfs.to_feather(Path.cwd().parent.parent / "data" / "AGFD_OT_V2.feather")

In [ ]:
dfs[
    (dfs["class"] == "crack")
    & (dfs["speed"] == 500)
    & (dfs["load"] == 11)
    # & (dfs["severity"] == "mild")
    # & (dfs["installation"] == 2)
    & (dfs["OT_method"] == "signal")
]

## Error checking


In [ ]:
df_test = pd.read_feather(Path.cwd().parent.parent / "data" / "AGFD_OT.feather")
df_test[
    (df_test["class"] == "pitting")
    & (df_test["speed"] == 750)
    & (df_test["load"] == 1)
    & (df_test["installation"] == 3)
    & (df_test["OT_method"] == "signal")
]

In [ ]:
# Change the class of all entries where class is "crack" and severity is "L" to "missing_tooth"
df_test.loc[(df_test["class"] == "crack") & (df_test["severity"] == "L"), "class"] = (
    "missing_tooth"
)

In [ ]:
df_test["class"].value_counts()

In [ ]:
df_test[df_test["class"] == "missing_tooth"]

In [ ]:
px.line(df_test[df_test["class"] == "missing_tooth"].iloc[0]["acc4"])

In [ ]:
df_test = pd.read_csv(
    Path.cwd().parent.parent.parent
    / "data"
    / "mendeley_AGFD3kHz"
    / "1500RPM"
    / "0_12Nm"
    / "Faulty"
    / "Inst_1"
    / "Mild_Pitting_GP1.csv",
    sep=",",
    index_col=0,
    header=0,
)
t = df_test["time"].to_numpy()  # [-100000:]
# angle = -df_test["enc4_ang"].to_numpy()  # [-100000:]
signal = df_test["acc4"].to_numpy()  # [-100000:]

In [ ]:
np.sum(np.isnan(signal))

In [ ]:
t[-3:]

## Time skipping showcase


In [ ]:
df = pd.read_csv(
    Path.cwd().parent.parent.parent
    / "data"
    / "mendeley_AGFD3kHz"
    / "1500RPM"
    / "0_12Nm"
    / "Healthy"
    / "GP1.csv",
    sep=",",
    index_col=0,
    header=0,
)
t = df["time"].to_numpy()  # [-100000:]
angle = -df["enc4_ang"].to_numpy()  # [-100000:]
# signal = df["acc2"].to_numpy()  # [-100000:]

In [ ]:
crossings = np.where(np.logical_and(angle[:-1] % 360 <= 5, angle[1:] % 360 > 5))[0]
px.line(np.diff(crossings[int(len(crossings) * 0.7) : int(len(crossings) * 0.9)]))

In [ ]:
px.line(np.diff(t[int(len(t) * 0.7) : int(len(t) * 0.9)]))

## Visualization


In [ ]:
df = pd.read_csv(
    Path.cwd().parent.parent.parent
    / "data"
    / "mendeley_AGFD3kHz"
    / "500RPM"
    / "1_31Nm"
    / "Healthy"
    / "GP9.csv",
    sep=",",
    index_col=0,
    header=0,
)
t = df["time"].to_numpy()
angle = -df["enc4_ang"].to_numpy()
signal = df["Torq1"].to_numpy()

plt.figure(figsize=(15, 5))
plt.plot(signal[-1000:])

In [ ]:
df = pd.read_csv(
    Path.cwd().parent.parent.parent
    / "data"
    / "mendeley_AGFD3kHz"
    / "500RPM"
    / "1_31Nm"
    / "Faulty"
    / "Inst_2"
    / "Mild_TFF_GP9.csv",
    sep=",",
    index_col=0,
    header=0,
)
t = df["time"].to_numpy()
angle = -df["enc4_ang"].to_numpy()
signal = df["Torq2"].to_numpy()

plt.figure(figsize=(15, 5))
plt.plot(signal[-10000:])

signal.mean()

## Clustering Visualizations


In [ ]:
# Crack
A1 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "crack")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "S")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
A2 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "crack")
        & (df_test["load"] == 11)
        & (df_test["severity"] == "S")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
A3 = np.stack(
    df_test[
        (df_test["speed"] == 500)
        & (df_test["class"] == "crack")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "S")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
A4 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "crack")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "L")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
A5 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "crack")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "S")
        & (df_test["installation"] == 2)
    ]["acc3"]
)

# Wear
B1 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "wear")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "L")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
B2 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "wear")
        & (df_test["load"] == 11)
        & (df_test["severity"] == "L")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
B3 = np.stack(
    df_test[
        (df_test["speed"] == 500)
        & (df_test["class"] == "wear")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "L")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
B4 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "wear")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "S")
        & (df_test["installation"] == 1)
    ]["acc3"]
)
B5 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "wear")
        & (df_test["load"] == 6)
        & (df_test["severity"] == "L")
        & (df_test["installation"] == 2)
    ]["acc3"]
)

# Healthy/baseline
C1 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "healthy")
        & (df_test["load"] == 6)
        & (df_test["healthy_GP"] == 9)
    ]["acc3"]
)
C2 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "healthy")
        & (df_test["load"] == 11)
        & (df_test["healthy_GP"] == 9)
    ]["acc3"]
)
C3 = np.stack(
    df_test[
        (df_test["speed"] == 500)
        & (df_test["class"] == "healthy")
        & (df_test["load"] == 6)
        & (df_test["healthy_GP"] == 9)
    ]["acc3"]
)
C4 = np.stack(
    df_test[
        (df_test["speed"] == 1000)
        & (df_test["class"] == "healthy")
        & (df_test["load"] == 6)
        & (df_test["healthy_GP"] == 2)
    ]["acc3"]
)

A1 = normalize(A1, norm="l2", axis=1)
A2 = normalize(A2, norm="l2", axis=1)
A3 = normalize(A3, norm="l2", axis=1)
A4 = normalize(A4, norm="l2", axis=1)
A5 = normalize(A5, norm="l2", axis=1)
B1 = normalize(B1, norm="l2", axis=1)
B2 = normalize(B2, norm="l2", axis=1)
B3 = normalize(B3, norm="l2", axis=1)
B4 = normalize(B4, norm="l2", axis=1)
B5 = normalize(B5, norm="l2", axis=1)
C1 = normalize(C1, norm="l2", axis=1)
C2 = normalize(C2, norm="l2", axis=1)
C3 = normalize(C3, norm="l2", axis=1)
C4 = normalize(C4, norm="l2", axis=1)

In [ ]:
pca = PCA(n_components=3)
pca_embeddings = pca.fit_transform(
    np.concatenate(
        [
            A1,
            A2,
            A3,
            A4,
            A5,
            B1,
            B2,
            B3,
            B4,
            B5,
            C1,
            C2,
            C3,
            C4,
        ],
        axis=0,
    )
)
pca_embeddings = pd.DataFrame(pca_embeddings, columns=["x", "y", "z"])

pca_embeddings["class"] = np.concatenate([
    len(A1) * ["Crack"],
    len(A2) * ["Crack"],
    len(A3) * ["Crack"],
    len(A4) * ["Crack"],
    len(A5) * ["Crack"],
    len(B1) * ["Wear"],
    len(B2) * ["Wear"],
    len(B3) * ["Wear"],
    len(B4) * ["Wear"],
    len(B5) * ["Wear"],
    len(C1) * ["Baseline"],
    len(C2) * ["Baseline"],
    len(C3) * ["Baseline"],
    len(C4) * ["Baseline"],
])
pca_embeddings["rpm"] = np.concatenate([
    len(A1) * [1000],
    len(A2) * [1000],
    len(A3) * [500],
    len(A4) * [1000],
    len(A5) * [1000],
    len(B1) * [1000],
    len(B2) * [1000],
    len(B3) * [500],
    len(B4) * [1000],
    len(B5) * [1000],
    len(C1) * [1000],
    len(C2) * [500],
    len(C3) * [1000],
    len(C4) * [1000],
])
pca_embeddings["torque"] = np.concatenate([
    len(A1) * [6],
    len(A2) * [11],
    len(A3) * [6],
    len(A4) * [6],
    len(A5) * [6],
    len(B1) * [6],
    len(B2) * [11],
    len(B3) * [6],
    len(B4) * [6],
    len(B5) * [6],
    len(C1) * [6],
    len(C2) * [11],
    len(C3) * [6],
    len(C4) * [6],
])
pca_embeddings["installation"] = np.concatenate([
    len(A1) * [1],
    len(A2) * [1],
    len(A3) * [1],
    len(A4) * [1],
    len(A5) * [3],
    len(B1) * [1],
    len(B2) * [1],
    len(B3) * [1],
    len(B4) * [1],
    len(B5) * [3],
    len(C1) * [9],
    len(C2) * [9],
    len(C3) * [9],
    len(C4) * [2],
])
pca_embeddings["level"] = np.concatenate([
    len(A1) * ["mild"],
    len(A2) * ["mild"],
    len(A3) * ["mild"],
    len(A4) * ["severe"],
    len(A5) * ["mild"],
    len(B1) * ["severe"],
    len(B2) * ["severe"],
    len(B3) * ["severe"],
    len(B4) * ["mild"],
    len(B5) * ["-"],
    len(C1) * ["-"],
    len(C2) * ["-"],
    len(C3) * ["-"],
    len(C4) * ["-"],
])

fig = px.scatter_3d(
    pca_embeddings,
    x="x",
    y="y",
    z="z",
    color="class",
    symbol="rpm",
    hover_data=["class", "rpm", "torque", "installation", "level"],
    width=1000,
    height=800,
)

fig.update_layout(
    title=dict(text="PCA", font=dict(size=50), automargin=True, yref="paper")
)
fig.show()

In [ ]:
tsne_embeddings = TSNE(
    n_components=3, learning_rate="auto", init="random", perplexity=20
).fit_transform(
    np.concatenate(
        [
            A1,
            A2,
            A3,
            A4,
            A5,
            B1,
            B2,
            B3,
            B4,
            B5,
            C1,
            C2,
            C3,
            C4,
        ],
        axis=0,
    )
)
tsne_embeddings = pd.DataFrame(tsne_embeddings, columns=["x", "y", "z"])

tsne_embeddings["class"] = np.concatenate([
    len(A1) * ["Crack"],
    len(A2) * ["Crack"],
    len(A3) * ["Crack"],
    len(A4) * ["Crack"],
    len(A5) * ["Crack"],
    len(B1) * ["Wear"],
    len(B2) * ["Wear"],
    len(B3) * ["Wear"],
    len(B4) * ["Wear"],
    len(B5) * ["Wear"],
    len(C1) * ["Baseline"],
    len(C2) * ["Baseline"],
    len(C3) * ["Baseline"],
    len(C4) * ["Baseline"],
])
tsne_embeddings["rpm"] = np.concatenate([
    len(A1) * [1000],
    len(A2) * [1000],
    len(A3) * [500],
    len(A4) * [1000],
    len(A5) * [1000],
    len(B1) * [1000],
    len(B2) * [1000],
    len(B3) * [500],
    len(B4) * [1000],
    len(B5) * [1000],
    len(C1) * [1000],
    len(C2) * [500],
    len(C3) * [1000],
    len(C4) * [1000],
])
tsne_embeddings["torque"] = np.concatenate([
    len(A1) * [6],
    len(A2) * [11],
    len(A3) * [6],
    len(A4) * [6],
    len(A5) * [6],
    len(B1) * [6],
    len(B2) * [11],
    len(B3) * [6],
    len(B4) * [6],
    len(B5) * [6],
    len(C1) * [6],
    len(C2) * [11],
    len(C3) * [6],
    len(C4) * [6],
])
tsne_embeddings["installation"] = np.concatenate([
    len(A1) * [1],
    len(A2) * [1],
    len(A3) * [1],
    len(A4) * [1],
    len(A5) * [3],
    len(B1) * [1],
    len(B2) * [1],
    len(B3) * [1],
    len(B4) * [1],
    len(B5) * [3],
    len(C1) * [9],
    len(C2) * [9],
    len(C3) * [9],
    len(C4) * [2],
])
tsne_embeddings["level"] = np.concatenate([
    len(A1) * ["mild"],
    len(A2) * ["mild"],
    len(A3) * ["mild"],
    len(A4) * ["severe"],
    len(A5) * ["mild"],
    len(B1) * ["severe"],
    len(B2) * ["severe"],
    len(B3) * ["severe"],
    len(B4) * ["mild"],
    len(B5) * ["-"],
    len(C1) * ["-"],
    len(C2) * ["-"],
    len(C3) * ["-"],
    len(C4) * ["-"],
])

fig = px.scatter_3d(
    tsne_embeddings,
    x="x",
    y="y",
    z="z",
    color="class",
    symbol="rpm",
    hover_data=["class", "rpm", "torque", "installation", "level"],
    width=1000,
    height=800,
)

fig.update_layout(
    title=dict(text="PCA", font=dict(size=50), automargin=True, yref="paper")
)
fig.show()

In [ ]:
a = pd.DataFrame({"A": range(3), "B": range(1, 4), "C": range(2, 5), "D": range(3, 6)})
# a[["B", "C"]] = a[["B", "C"]].transform(lambda x: x + 10)
pd.melt(a, id_vars=["A"], value_vars=["C", "D"], var_name="var", value_name="val")
# a

In [ ]:
a = np.arange(100 * 65).reshape(100, 65)
a / a[:, 1:2]

### Cepstrum


In [ ]:
df_test

In [ ]:
df_test[(df_test["class"] == "healthy")]

In [ ]:
spectrum = df_test[
    (df_test["class"] == "crack")
    & (df_test["severity"] == "L")
    & (df_test["speed"] == 1000)
    & (df_test["load"] == 11)
    & (df_test["OT_method"] == "signal")
    & (df_test["installation"] == 1)
].iloc[0]["acc2"]
plt.plot(spectrum)

In [ ]:
spectrum_healthy = df_test[
    (df_test["class"] == "healthy")
    & (df_test["speed"] == 1000)
    & (df_test["load"] == 11)
    & (df_test["OT_method"] == "signal")
    & (df_test["healthy_GP"] == 5)
].iloc[0]["acc2"]
plt.plot(spectrum)

spectrum_healthy2 = df_test[
    (df_test["class"] == "healthy")
    & (df_test["speed"] == 1000)
    & (df_test["load"] == 11)
    & (df_test["OT_method"] == "signal")
    & (df_test["healthy_GP"] == 6)
].iloc[0]["acc2"]

plt.plot(spectrum_healthy)
plt.plot(spectrum_healthy2)

In [ ]:
ceps_healthy = np.fft.ifft(np.log(spectrum_healthy)).real
ceps_healthy2 = np.fft.ifft(np.log(spectrum_healthy2)).real
ceps = np.fft.ifft(np.log(spectrum)).real
plt.plot(ceps_healthy[1:])
plt.plot(ceps[1:])

In [ ]:
plt.plot(ceps[1:] - ceps_healthy[1:])

In [ ]:
plt.plot(ceps_healthy2[1:] - ceps_healthy[1:])